`Qwen/Qwen3-0.6B`: post-trained instruction-following checkpoint, with non-thinking chat formatting. 


In [ ]:
%pip install "transformers==5.17.0"

In [ ]:
import platform
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Select a GPU runtime in Colab before continuing."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model_id = "Qwen/Qwen3-0.6B"
# fixes the HF checkpoint and tokenizer
model_revision = "c1899de289a04d12100db370d81485cdf75e47ca"
device = torch.device("cuda")
dtype = torch.float32

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    revision=model_revision,
)

In [ ]:
# load model from HF
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    revision=model_revision,
    dtype=dtype,
    # non optimized, PyTorhc attention
    attn_implementation="eager",
).to(device)
# no grad retention
model.eval()
model.requires_grad_(False)

print(type(model).__name__)
print("Blocks:", model.config.num_hidden_layers)
print("Residual width:", model.config.hidden_size)
print("Vocabulary dimension:", model.config.vocab_size)

In [ ]:
# run a single forward pass
def forward_tokens(token_ids):
    # embed the token IDs: [B, T] -> [B, T, d_model]
    embeddings = model.get_input_embeddings()(token_ids)
    # inference: [B, T, d_model] -> [B, T, VocabSize]
    outputs = model(
        inputs_embeds=embeddings,
        use_cache=False,
        logits_to_keep=0,
    )
    return token_ids, embeddings, outputs.logits

# wrapper for string prompts
def read_prompt(text):
    # tokenize according to loaded tokenizer
    token_ids = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids.to(device)
    if token_ids.shape[1] == 0:
        raise ValueError("The text must contain at least one token.")
    # pass tokens [B, T] to forward pass -> logits [B, T, VocabSize]
    return forward_tokens(token_ids)

In [ ]:
question = "What is the capital of France? Answer with just the city."
prompt_text = tokenizer.apply_chat_template(
    [{"role": "user", "content": question}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print(repr(prompt_text))

In [ ]:
# run the forward pass
with torch.no_grad():
    token_ids, embeddings, logits = read_prompt(prompt_text)

print("token_ids [batch, position]:", tuple(token_ids.shape))
print("embeddings [batch, position, coordinate]:", tuple(embeddings.shape))
print("logits [batch, position, vocabulary]:", tuple(logits.shape))
print("IDs:", token_ids[0].tolist())
print("Pieces:", tokenizer.convert_ids_to_tokens(token_ids[0].tolist()))

In [ ]:
# compute softmax with temperature
def temperature_softmax(logits, temperature=1.0):
    if not temperature > 0:
        raise ValueError("Temperature must be positive.")
    return torch.softmax(logits / temperature, dim=-1)

In [ ]:
# set temp
temperature = 0.7
# extract last logit vector [1, VocabSize]
next_logits = logits[0, -1, :]
# -> probabilities
next_probabilities = temperature_softmax(next_logits, temperature)
# pull top 10
top_probabilities, top_ids = next_probabilities.topk(10)

print("Probability sum:", next_probabilities.sum().item())
for token_id, probability in zip(top_ids.tolist(), top_probabilities.tolist()):
    print(token_id, repr(tokenizer.decode([token_id])), probability)

In [ ]:
# take in tokens [B, T] -> [B, T + max_new_tokens]
def sample_tokens(token_ids, max_new_tokens=20, temperature=0.7):
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be nonnegative.")
    # copy tokens locally (will be appended-to)
    sequence = token_ids.clone()
    # get the config-specified index for EOS token (to see when model's done)
    eos_token_ids = model.generation_config.eos_token_id
    for step in range(max_new_tokens):
        # get a single forward pass's logits
        _, _, step_logits = forward_tokens(sequence)
        # -> probabilities
        probabilities = temperature_softmax(step_logits[:, -1, :], temperature)
        # sample the distribution
        next_token = torch.multinomial(probabilities, num_samples=1)
        # append sampled token ID to sequence for next pass
        sequence = torch.cat((sequence, next_token), dim=1)
        if next_token.item() in eos_token_ids:
            break
    return sequence

In [ ]:
torch.manual_seed(0)
with torch.no_grad():
    # call autoregressive sampling
    sequence = sample_tokens(token_ids, max_new_tokens=20, temperature=0.7)

# what are the new token IDs
new_ids = sequence[0, token_ids.shape[1]:]
print("New IDs:", new_ids.tolist())
# literal decoded tokens
print("Literal continuation:", repr(tokenizer.decode(new_ids.tolist())))
# human-readable decoded tokens
print("Answer:", tokenizer.decode(new_ids.tolist(), skip_special_tokens=True))

TO BE IMPLEMENTED

In [ ]:
def read_parameters():
    raise NotImplementedError("To implement together after the inference cells.")

In [ ]:
def read_activations(token_ids, layer):
    raise NotImplementedError("To implement together after the inference cells.")